# Part 3 — Planning the Work: plan-and-execute, ReWOO, and the tool DAG

*Agents from First Principles, Part 3.*

The agent from Parts 1 and 2 is a **ReAct loop**: it decides ONE step, runs it, reads the result, and only then decides the next step. That reactivity is its strength -- and its bill. Every hop is a fresh LLM call, and every call re-sends the **whole transcript so far**. On a task with four lookups that is five LLM calls, each one longer than the last, and the model re-derives the plan from scratch every step, so it wanders.

The fix is to stop treating the plan as something that lives only in the model's head between calls, and make the **plan a first-class artifact**: write the whole plan down once, then execute it. Three ways to do that, each cheaper or more parallel than the last:

1. **Plan-and-Execute** -- a planner writes an ordered plan once; an executor runs the steps *without asking the model again*; one final call synthesizes. **2 LLM calls** instead of one-per-hop. (It can also REPLAN when a step surprises it -- that hook is Part 4.)
2. **ReWOO** (Reasoning WithOut Observation) -- the planner writes the plan with **evidence variables** (`#E1`, `#E2`, ...) so a later step can name an earlier step's result *before it exists*. The tools run and fill the variables in; one solver call reads the filled-in worksheet. Still 2 LLM calls, and the plan never pauses to consult the model mid-run.
3. **Tool DAG** (LLMCompiler-style) -- the planner emits a **DAG** of tasks with explicit dependencies. Independent tasks sit at the same depth and run in one round; only a real dependency forces a new round. The headline number is the **critical-path depth** (the longest chain of dependent steps), not the step count.

**How we measure, and what we refuse to fake.** Two honest numbers per strategy: the **LLM call count** (the dominant cost lever -- every invocation of the model is paid for) and the **critical-path depth** (the number of sequential rounds of tool calls, i.e. the longest dependency chain; independent calls in the DAG share a round). We deliberately do NOT print wall-clock: the offline runner executes everything sequentially, "parallel" here means "could run in the same round," and the depth is what real concurrency would shrink. (The transcript-resend cost ReAct pays is real too; we flag it here, and Part 11 develops transcript economics in full.)

**The task is built so the differences SHOW.** One true dependency chain and two independent branches: the refund window (a policy lookup, independent), the earbuds warranty (TWO dependent hops: who acquired Acme, then *their* warranty), and 18% tax on a $250 order (a calculation, independent). Four tool calls, same correct answer every way. Only the cost and the depth differ.

**Continuity**: same world and tools as Parts 1-2 (refund policy, the Acme to Globex chain, the calculator). Deterministic rule planner plus controller offline; `generate()` is the real-LLM path one env flag away.

> **Runs with no network, no API key, and no third-party dependency.** Set `OPENAI_API_KEY` to see the real-planner banner; every strategy always falls through to the deterministic rule planner so output stays reproducible.

Companion script: `planners.py`

## Setup

Two standard-library imports do the work: `os` lets us check for an API key without ever requiring one, and `re` powers the lexical retriever's tokenizer, the calculator's input guard, and the acquirer parser. Nothing in the default path is imported from a third-party package, so every cell runs fully offline.

In [ ]:
import os
import re

print("ready -- no network, no API key, no third-party dependency required")

## Step 0 -- The world and tools, carried from Parts 1-2

Nothing here is new; it just keeps the notebook self-contained. The two corpora and the lexical retriever are exactly the ones Parts 1-2 used. `POLICY_KB` is the support corpus (refunds, the E-4042 error, shipping, warranty); `PRODUCTS` is the Acme-to-Globex acquisition chain, split so no single chunk holds both "who acquired Acme" and "the earbuds warranty" -- that gap is the one true dependency in the task. The retriever scores each chunk by content-word overlap, normalized by the geometric mean of the two token counts: a cosine-flavored score in [0, 1], deterministic and model-free so the output is reproducible. The three read-only tools (`search_policy`, `search_products`, `calculator`) and the `call_tool` dispatcher are carried too; `_acquirer_from` pulls the acquirer's name out of a retrieved chunk. **The new material starts at Step 1.**

In [ ]:
POLICY_KB = [
    "Refunds are accepted within 30 days of purchase, provided the item is unused and in its original packaging.",
    "To start a return, email support@example.com with your order number. Refunds are processed within five business days of us receiving the item.",
    "Error E-4042 means the payment was declined by the bank; ask the customer to retry with a different card or contact their bank.",
    "Standard shipping takes 3 to 5 business days. Express shipping arrives the next business day.",
    "All electronics include a one-year limited warranty covering manufacturing defects.",
]

PRODUCTS = [
    "Acme Corp was acquired by Globex in 2024.",
    "Globex now manufactures the wireless earbuds product line it inherited from Acme.",
    "Globex-branded wireless earbuds carry a 2-year limited warranty.",
    "The wireless earbuds deliver up to 8 hours of battery life, and up to 24 hours with the charging case.",
]

_TOKEN_RE = re.compile(r"[a-z0-9]+")
_STOPWORDS = {
    "what", "is", "are", "the", "of", "a", "an", "for", "on", "in", "to", "how",
    "do", "does", "and", "my", "i", "there", "with", "your", "our", "who", "by",
    "that", "made", "company", "s", "whats", "percent",
}


def _stem(tok):
    return tok[:-1] if len(tok) > 3 and tok.endswith("s") else tok


def _tokens(text):
    return [_stem(t) for t in _TOKEN_RE.findall(text.lower()) if t not in _STOPWORDS]


class _LexicalRetriever:
    """Deterministic, model-free stand-in for a dense retriever (RAG Part 6)."""

    def __init__(self, corpus):
        self.chunks = list(corpus)
        self._chunk_tokens = [set(_tokens(c)) for c in self.chunks]

    def _score(self, q_tokens, c_tokens):
        if not q_tokens or not c_tokens:
            return 0.0
        overlap = len(q_tokens & c_tokens)
        return overlap / ((len(q_tokens) * len(c_tokens)) ** 0.5)

    def retrieve(self, query, k=1):
        q = set(_tokens(query))
        scored = [(self.chunks[i], self._score(q, self._chunk_tokens[i]))
                  for i in range(len(self.chunks))]
        scored.sort(key=lambda x: -x[1])
        return scored[:k]


_POLICY_STORE = _LexicalRetriever(POLICY_KB)
_PRODUCTS_STORE = _LexicalRetriever(PRODUCTS)
_CALC_RE = re.compile(r"^[\d\s+\-*/().%]+$")


def search_policy(query):
    return _POLICY_STORE.retrieve(query, k=1)[0]      # (text, score)


def search_products(query):
    return _PRODUCTS_STORE.retrieve(query, k=1)[0]    # (text, score)


def calculator(expression):
    if not _CALC_RE.match(expression):
        return "calculator error"
    try:
        return eval(expression, {"__builtins__": {}}, {})
    except Exception as exc:
        return f"calculator error: {type(exc).__name__}"


def call_tool(tool, arg):
    """Run a tool, return (observation_text, score_or_None)."""
    if tool == "search_policy":
        text, score = search_policy(arg)
        return text, score
    if tool == "search_products":
        text, score = search_products(arg)
        return text, score
    if tool == "calculator":
        return calculator(arg), None
    return f"[unknown tool {tool}]", None


def _acquirer_from(text):
    m = re.search(r"acquired by (\w+)", text)
    return m.group(1) if m else "the acquirer"


print(f"Policy KB: {len(POLICY_KB)} chunks. Products: {len(PRODUCTS)} chunks. Tools + retriever ready (carried from Parts 1-2).")

## Step 1 -- A meter, because the whole point of this part is counting

Every claim in this notebook is a count, so we count explicitly rather than hand-wave. `CallMeter` tracks the two numbers that matter: **LLM calls** (the dominant cost lever -- every invocation of the model is paid for) and **tool calls**. It also accumulates `transcript_chars`, the cumulative context re-sent to the model, so ReAct's hidden tax (it re-sends the whole transcript on every hop) shows up as a real number instead of a vibe. Each strategy threads one meter through its run; the scoreboard at the end reads them back.

In [ ]:
class CallMeter:
    def __init__(self):
        self.llm = 0
        self.tool = 0
        self.transcript_chars = 0     # cumulative context re-sent to the model

    def llm_call(self, context_chars=0):
        self.llm += 1
        self.transcript_chars += context_chars

    def tool_call(self):
        self.tool += 1


print("CallMeter ready: counts LLM calls (the cost lever), tool calls, and transcript chars re-sent.")

## Step 2 -- The plan as data: `TaskNode`, the planner, binding, and the DAG layering

This is the net-new idea of the part. A `TaskNode` is one planned tool call with an id (`E1`, `E2`, ...), the tool and its argument, and an explicit list of dependency ids. This single structure backs all three planners. **ReAct does NOT use it** -- it has no plan object at all, and that absence is exactly the point.

An **evidence variable** like `#E2` sitting inside an argument names another node's result *before that node has run*. `_bind` resolves those placeholders once the referenced node has a result: here a reference to `E2` binds to the **acquirer** named in `E2`'s result, so `'#E2 earbuds warranty'` becomes `'Globex earbuds warranty'`. (A real ReWOO binds the raw text and lets the solver parse it; we bind the clean name so the demo query reads naturally.)

`rule_planner` is the deterministic offline planner -- a real system swaps its body for one `generate()` call returning the same structure. It emits **four** tool nodes; the only dependency is `E3` (the warranty) needing `E2` (the acquirer's name). `compose_answer` synthesizes the final answer from whatever evidence was gathered, so identical inputs give an identical answer -- every strategy returns the same thing, only the cost differs. `dag_levels` computes the **critical-path depth** by longest-path layering: a node's level is one more than its deepest dependency, and the maximum level is the number of sequential rounds.

In [ ]:
class TaskNode:
    def __init__(self, eid, tool, arg, deps):
        self.eid = eid          # "E1", "E2", ...
        self.tool = tool
        self.arg = arg          # may contain "#En" placeholders
        self.deps = deps        # list of eids this node needs first
        self.result = None      # filled at execution
        self.score = None


GOAL = ("Build a refund summary: the refund window from policy, the warranty on "
        "the earbuds made by the company that acquired Acme, and 18% tax on a "
        "$250 order.")


def rule_planner(goal):
    """The deterministic offline planner. A real system swaps this body for one
    generate() call that returns the same structure. It emits FOUR tool nodes; the
    only dependency is E3 (warranty) needing E2 (the acquirer's name)."""
    return [
        TaskNode("E1", "search_policy", "refund window 30 days", deps=[]),
        TaskNode("E2", "search_products", "who acquired Acme", deps=[]),
        TaskNode("E3", "search_products", "#E2 earbuds warranty", deps=["E2"]),
        TaskNode("E4", "calculator", "0.18 * 250", deps=[]),
    ]


def _bind(arg, evidence):
    """Resolve #En placeholders in an arg using already-computed evidence. Here a
    reference to E2 binds to the ACQUIRER named in E2's result (a real ReWOO binds
    the raw text and lets the solver parse it; we bind the clean name so the demo
    query reads naturally)."""
    out = arg
    for eid, node in evidence.items():
        if "#" + eid in out and node.result is not None:
            out = out.replace("#" + eid, _acquirer_from(node.result))
    return out


def compose_answer(evidence):
    """Synthesize the final answer from the evidence. Identical inputs -> identical
    answer, so every strategy returns the same thing; only the cost differs."""
    window = "30 days from purchase" if "30 days" in evidence["E1"].result else "see policy"
    acquirer = _acquirer_from(evidence["E2"].result)
    warranty = evidence["E3"].result
    term = "2-year limited warranty" if "2-year" in warranty else warranty
    tax = evidence["E4"].result
    return (f"Refund window is {window}. The earbuds (made by {acquirer}, which "
            f"acquired Acme) carry a {term}. Tax on a $250 order at 18% is ${tax:.2f}.")


def dag_levels(nodes):
    """Critical-path depth via longest-path layering. A node's level is one more
    than the deepest dependency; the max level is the number of sequential rounds."""
    by_id = {n.eid: n for n in nodes}
    level = {}

    def lvl(eid):
        if eid in level:
            return level[eid]
        node = by_id[eid]
        level[eid] = 1 + max([lvl(d) for d in node.deps], default=0)
        return level[eid]

    for n in nodes:
        lvl(n.eid)
    return level


print("TaskNode, rule_planner, _bind, compose_answer, dag_levels ready.")
print("Plan:", [(n.eid, n.tool, n.deps) for n in rule_planner(GOAL)])

## Step 3 -- `generate()`: the real LLM path (reference shape only)

Same device as Parts 1-2. In production the planner *is* an LLM: you hand it the goal and the rendered action space and it returns a plan in the same `TaskNode` shape, and a second call synthesizes the answer from the filled-in evidence. `generate()` is the single swappable call that lights up the real path; the offline demo never calls it. The deterministic `rule_planner` (Step 2) is the source of truth for every plan you see in the output. SDK names and model IDs move fast -- only `generate()` would need edits to go live.

In [ ]:
def generate(prompt):
    """REAL path: ask a hosted LLM to plan or synthesize. Unused offline."""
    from openai import OpenAI
    client = OpenAI()                               # reads OPENAI_API_KEY
    resp = client.chat.completions.create(
        model="gpt-4o-mini",                        # a small chat model; check names
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content

# Anthropic / Claude alternative. Swap in for generate() above:
#
# def generate(prompt):
#     from anthropic import Anthropic
#     client = Anthropic()                            # reads ANTHROPIC_API_KEY
#     resp = client.messages.create(
#         model="claude-opus-4-8",                    # check current model names
#         max_tokens=1024,
#         messages=[{"role": "user", "content": prompt}],
#     )
#     return resp.content[0].text


if os.environ.get("OPENAI_API_KEY"):
    print("[planner] OPENAI_API_KEY set; the real LLM planner would emit the same "
          "plan structure via generate(). Falling through to the deterministic rule "
          "planner so output stays reproducible.")
else:
    print("[planner] no OPENAI_API_KEY; using deterministic rule planner/controller "
          "(offline default)")

## Step 4 -- Strategy A: ReAct (Parts 1-2), the baseline we are trying to beat

ReAct decides one step at a time. Each step is a **fresh LLM call** that re-reads the whole transcript, and no plan is ever written down. On this four-lookup task that is **five** LLM calls: one decision per hop, plus one final call to read the transcript and write the answer. The depth is fully serial -- one round per hop -- because ReAct cannot see that the policy lookup and the tax calculation have nothing to do with the acquirer chain; it only knows the next step after it has read the last observation. The meter also records the transcript it re-sends each hop, the hidden tax we develop fully in Part 11.

In [ ]:
def run_react(goal):
    print("  ReAct decides one step at a time; each step is a fresh LLM call that")
    print("  re-reads the whole transcript. No plan is ever written down.")
    meter = CallMeter()
    transcript = []                                   # observations so far
    order = [("search_policy", "refund window 30 days"),
             ("search_products", "who acquired Acme"),
             ("search_products", "{acquirer} earbuds warranty"),
             ("calculator", "0.18 * 250")]
    evidence = {}
    acquirer = None
    for i, (tool, arg) in enumerate(order, start=1):
        ctx = "\n".join(transcript)
        meter.llm_call(context_chars=len(ctx))        # the per-step decision call
        if "{acquirer}" in arg:
            arg = arg.replace("{acquirer}", acquirer or "the acquirer")
        obs, score = call_tool(tool, arg)
        meter.tool_call()
        if tool == "search_products" and "acquired by" in obs:
            acquirer = _acquirer_from(obs)
        evidence[f"step{i}"] = obs
        shown = f"{obs} (score={score:.2f})" if score is not None else f"{obs}"
        print(f"    step {i}: LLM picks {tool}({arg!r}) -> {shown}")
        transcript.append(f"{tool}({arg}) -> {obs}")
    ctx = "\n".join(transcript)
    meter.llm_call(context_chars=len(ctx))            # the final finish/answer call
    print(f"    step 5: LLM reads the full transcript and writes the answer.")
    answer = ("Refund window is 30 days from purchase. The earbuds (made by Globex, "
              "which acquired Acme) carry a 2-year limited warranty. Tax on a $250 "
              "order at 18% is $45.00.")
    depth = len(order)                                # fully serial: one round per hop
    print(f"    -> {meter.llm} LLM calls, {meter.tool} tool calls, depth {depth} "
          f"(serial). Transcript re-sent: ~{meter.transcript_chars} chars total.")
    return answer, meter, depth


print("run_react ready.")

## Step 5 -- Strategy B: Plan-and-Execute

The first strategy that treats the plan as an artifact. **One** planning call writes the ordered list of `TaskNode`s; an executor then runs the list step by step **without ever consulting the model**; **one** synthesis call composes the answer at the end. That is two LLM calls regardless of how many tools the plan contains -- the model cost is now decoupled from the number of hops. The depth is still 4 here because this executor is linear (one round per step), so it does not yet exploit the two independent branches; the DAG in Step 7 does. Note `_bind` quietly resolving `#E2` inside `E3`'s argument as the executor reaches it.

In [ ]:
def run_plan_execute(goal):
    meter = CallMeter()
    meter.llm_call(context_chars=len(goal))           # ONE planning call
    plan = rule_planner(goal)
    print("  Plan (written once, then executed without consulting the model):")
    for n in plan:
        dep = f" [needs {','.join(n.deps)}]" if n.deps else ""
        print(f"    {n.eid}: {n.tool}({n.arg!r}){dep}")
    evidence = {}
    for n in plan:                                    # executor: no LLM per step
        arg = _bind(n.arg, evidence)
        obs, score = call_tool(n.tool, arg)
        meter.tool_call()
        n.result, n.score = obs, score
        evidence[n.eid] = n
    meter.llm_call(context_chars=200)                 # ONE synthesis call
    answer = compose_answer(evidence)
    depth = len(plan)                                 # linear executor: one round per step
    print(f"    -> {meter.llm} LLM calls, {meter.tool} tool calls, depth {depth} "
          f"(linear). The executor never called the model.")
    return answer, meter, depth


print("run_plan_execute ready.")

## Step 6 -- Strategy C: ReWOO (Reasoning WithOut Observation)

ReWOO's move is to bind **evidence variables** up front. The planner writes a worksheet where a later step names an earlier step's result by variable (`#E2`) *before that result exists*, so the plan is complete the moment it is written -- there is no need to pause and consult the model after each observation. The tools run, filling the variables in (`#E2` binds to the acquirer's name), and **one** solver call reads the completed worksheet and writes the answer. Still two LLM calls, but the distinction from plan-and-execute is sharp: there are **no mid-run model calls of any kind**. Watch the `bound E3` line where `'#E2 earbuds warranty'` becomes `'Globex earbuds warranty'`.

In [ ]:
def run_rewoo(goal):
    meter = CallMeter()
    meter.llm_call(context_chars=len(goal))           # ONE planning call (with #E vars)
    plan = rule_planner(goal)
    print("  Worksheet (the planner names results as #E variables before they exist):")
    for n in plan:
        print(f"    #{n.eid} = {n.tool}[{n.arg!r}]")
    evidence = {}
    for n in plan:
        arg = _bind(n.arg, evidence)                  # #E2 -> the acquirer's name
        obs, score = call_tool(n.tool, arg)
        meter.tool_call()
        n.result, n.score = obs, score
        evidence[n.eid] = n
        if arg != n.arg:
            print(f"    bound {n.eid}: {n.arg!r} -> {arg!r}")
    meter.llm_call(context_chars=300)                 # ONE solver call over the worksheet
    answer = compose_answer(evidence)
    depth = len(plan)
    print(f"    -> {meter.llm} LLM calls, {meter.tool} tool calls, depth {depth} "
          f"(linear). One planner call, one solver call, no model calls in between.")
    return answer, meter, depth


print("run_rewoo ready.")

## Step 7 -- Strategy D: the tool DAG (LLMCompiler-style)

The same plan, but the executor reads the dependencies and runs it by **topological level**. Independent nodes share a round; only a real dependency forces a new one. `dag_levels` lays the four nodes out: `E1`, `E2`, `E4` have no dependencies, so they all sit at level 1 and run in **round 1**; `E3` needs `E2`, so it lands at level 2, **round 2**. The critical-path depth is therefore **2**, not 4 -- the longest dependency chain is `E2 -> E3`, and the two independent branches collapse into the first round. This is the headline of the part: depth, not step count, is what real concurrency would shrink. We still report depth rather than wall-clock because the offline runner executes everything sequentially; "parallel" means "could run in the same round."

In [ ]:
def run_dag(goal):
    meter = CallMeter()
    meter.llm_call(context_chars=len(goal))           # ONE planning call (emits the DAG)
    plan = rule_planner(goal)
    level = dag_levels(plan)
    depth = max(level.values())
    by_level = {}
    for n in plan:
        by_level.setdefault(level[n.eid], []).append(n)
    print("  DAG by dependency level (nodes in the same round could run in parallel):")
    evidence = {}
    for d in sorted(by_level):
        ids = ", ".join(n.eid for n in by_level[d])
        kind = "parallel" if len(by_level[d]) > 1 else "single"
        print(f"    round {d} ({kind}): {ids}")
        for n in by_level[d]:                          # one round; order within is irrelevant
            arg = _bind(n.arg, evidence)
            obs, score = call_tool(n.tool, arg)
            meter.tool_call()
            n.result, n.score = obs, score
            evidence[n.eid] = n
    meter.llm_call(context_chars=300)                 # ONE join/synthesis call
    answer = compose_answer(evidence)
    print(f"    -> {meter.llm} LLM calls, {meter.tool} tool calls, critical-path "
          f"depth {depth} (E2 -> E3 is the only chain; E1, E2, E4 share round 1).")
    return answer, meter, depth


print("run_dag ready.")

## Demo -- run all four strategies on the one task, then the scoreboard

Everything below runs fully offline. We run the same goal through all four strategies in order (ReAct, then Plan-and-Execute, then ReWOO, then the Tool DAG), assert that all four return the **identical** answer -- the whole point is "same answer, less cost" -- and then print the scoreboard. Read the two number columns that matter: **LLM calls** is the cost lever (5 drops to 2 the moment the plan becomes an artifact), and **critical-path depth** is the sequential rounds (4 drops to 2 only when the executor runs the DAG by level).

In [ ]:
bar = "=" * 72
print(bar)
print("PLANNING THE WORK  -  ReAct vs plan-and-execute vs ReWOO vs the tool DAG")
print(bar)
if os.environ.get("OPENAI_API_KEY"):
    print("[planner] OPENAI_API_KEY set; the real LLM planner would emit the same "
          "plan structure via generate(). Falling through to the deterministic rule "
          "planner so output stays reproducible.")
else:
    print("[planner] no OPENAI_API_KEY; using deterministic rule planner/controller "
          "(offline default)")
print(f"\nGOAL: {GOAL}")
print("Four tool calls, one dependency chain (acquirer -> warranty), two independent")
print("branches (policy, tax). Same correct answer every way; only cost and depth differ.")

results = {}

print("\n" + "-" * 72)
print("STRATEGY A - ReAct (Parts 1-2): one LLM call per hop, no plan object.")
print("-" * 72)
a_ans, a_m, a_d = run_react(GOAL)
results["ReAct"] = (a_m, a_d)

print("\n" + "-" * 72)
print("STRATEGY B - Plan-and-Execute: plan once, execute the list, synthesize once.")
print("-" * 72)
b_ans, b_m, b_d = run_plan_execute(GOAL)
results["Plan-and-Execute"] = (b_m, b_d)

print("\n" + "-" * 72)
print("STRATEGY C - ReWOO: plan with #E evidence variables; one solver call at the end.")
print("-" * 72)
c_ans, c_m, c_d = run_rewoo(GOAL)
results["ReWOO"] = (c_m, c_d)

print("\n" + "-" * 72)
print("STRATEGY D - Tool DAG (LLMCompiler): run by dependency level; depth, not count.")
print("-" * 72)
d_ans, d_m, d_d = run_dag(GOAL)
results["Tool DAG"] = (d_m, d_d)

# All four must agree on the answer; the whole point is "same answer, less cost."
print("\n" + bar)
print("SAME ANSWER, EVERY STRATEGY:")
print(f"  {a_ans}")
assert a_ans == b_ans == c_ans == d_ans, "strategies disagreed on the answer"

print("\n" + bar)
print("THE SCOREBOARD  (LLM calls is the cost lever; depth is the sequential rounds)")
print(bar)
print(f"  {'strategy':<20}{'LLM calls':>11}{'tool calls':>12}{'crit-path depth':>18}")
print("  " + "-" * 59)
for name in ["ReAct", "Plan-and-Execute", "ReWOO", "Tool DAG"]:
    m, d = results[name]
    print(f"  {name:<20}{m.llm:>11}{m.tool:>12}{d:>18}")
print("\n  Reading it:")
print("  - Writing the plan down once cuts LLM calls from 5 (one per hop) to 2")
print("    (plan + synthesize), and decouples model cost from the number of tools.")
print("  - ReWOO removes every mid-run model call via #E variable binding.")
print("  - The DAG additionally cuts critical-path depth from 4 to 2: the three")
print("    independent lookups collapse into one round; only acquirer -> warranty chains.")
print("  - We report depth, not wall-clock: the offline runner is sequential, and")
print("    depth is what real concurrency would shrink. (Transcript economics: Part 11.)")
print(bar)

## Wrap-up

Parts 1-2 built a ReAct loop that decides one step at a time. That reactivity costs a fresh LLM call per hop and re-sends the whole transcript each time, and it re-derives the plan every step. This part made the **plan a first-class artifact** and read three planners off it:

- **Plan-and-Execute** writes the plan once and executes it without the model, cutting LLM calls from 5 to 2 and decoupling model cost from the number of tools;
- **ReWOO** binds **evidence variables** (`#E2`) up front so the plan is complete before any tool runs, removing every mid-run model call;
- the **tool DAG** runs the plan by dependency level, so the two independent branches share a round and the **critical-path depth** drops from 4 to 2 -- depth, not step count, is what real concurrency shrinks.

Same answer every way; only the cost and the depth differ. We reported the two honest numbers (LLM calls and critical-path depth) and refused to fake a wall-clock speedup the sequential offline runner cannot prove; transcript economics get their full treatment in Part 11. **Part 4 -- Reflection and Replanning** is next: a plan written up front is only as good as the assumptions it was written under, so the agent learns to notice when a step surprises it and rewrite the rest of the plan mid-run.